# Nasiko request classifier: LLMRouter Longformer + KNN

This trains Nasiko's seven request types and ordinal complexity (1–5) from explicitly labeled, non-private examples. LLMRouter benchmark rows are **not labeled for Nasiko**: add them only after annotating the required schema below. Public/private scorer cases must never be added. The starter rows are development examples, not evidence of held-out performance.

In [16]:
%pip install llmrouter-lib==0.3.1 scikit-learn pandas numpy joblib


In [17]:
import hashlib, json, os, platform, time
from pathlib import Path
import joblib, numpy as np, pandas as pd, sklearn, torch
import torch
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import model_info
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (classification_report, f1_score, mean_absolute_error, accuracy_score)

LABELS = ['code_generation','code_understanding','technical_design','analytical_reasoning','writing','factual_lookup','general']
EMBEDDING_MODEL_ID = 'allenai/longformer-base-4096'
EMBEDDING_REVISION = os.environ.get('LONGFORMER_REVISION') or model_info(EMBEDDING_MODEL_ID).sha
assert len(EMBEDDING_REVISION) == 40, 'LONGFORMER_REVISION must be a resolved 40-character commit SHA'
TOKENIZER = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_ID, revision=EMBEDDING_REVISION)
ENCODER = AutoModel.from_pretrained(EMBEDDING_MODEL_ID, revision=EMBEDDING_REVISION).eval()
OUT = Path('llm-router/artifacts/request_classifier.joblib')


Loading weights:   0%|          | 0/271 [00:00<?, ?it/s]

[transformers] LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Labeled development data
Each `group` contains paraphrases/near duplicates and stays in one split. `source` identifies the source dataset; when at least three sources are available, the notebook holds out whole sources for validation and test. With fewer sources it falls back to paraphrase-group splits and explicitly reports the limitation. LLMRouter benchmark rows are not labeled for Nasiko: add them only after annotating this schema: `id, query, context(optional), request_type, complexity, group, source`. No private prompts or user data.


In [18]:
# Curated seed set: 30 code-generation, 25 code-understanding, and 20 examples per other type.
# Each row is a distinct prompt; add reviewed public benchmark examples through LLMROUTER_LABELLED_DATA.
rows = [
 ('code_generation','Write a Python function that merges two sorted lists.',2),
 ('code_generation','Implement a Rust parser for a small arithmetic grammar.',4),
 ('code_generation','Create a SQL migration adding an indexed nullable column.',3),
 ('code_generation','Implement a Go HTTP middleware that enforces a per-user token bucket.',4),
 ('code_generation','Write a TypeScript debounce helper with cancel and flush methods.',3),
 ('code_generation','Create a Bash script that rotates old application logs and reports failures.',3),
 ('code_generation','Implement a Java LRU cache with O(1) get and put operations.',4),
 ('code_generation','Add exponential backoff with jitter to this Python async retry loop.',4),
 ('code_generation','Build a C++ trie with insert, exact-match, and prefix-search methods.',4),
 ('code_generation','Create an accessible React dropdown that supports keyboard navigation.',4),
 ('code_generation','Write a Java stream pipeline that groups orders by status and totals their values.',3),
 ('code_generation','Implement a Python JSONL reader that processes records without loading the whole file.',3),
 ('code_generation','Write a Rust directory scanner that walks files concurrently and collects errors.',5),
 ('code_generation','Create a SQL query that reports each account monthly active days for the last year.',4),
 ('code_generation','Implement Dijkstra shortest paths in Python using an adjacency list and heap.',4),
 ('code_generation','Add pytest cases for valid expressions, malformed tokens, and nested parentheses.',3),
 ('code_generation','Define a TypeScript validator for a nested configuration object and its defaults.',3),
 ('code_generation','Implement cursor pagination in a TypeScript GraphQL resolver.',4),
 ('code_generation','Write a shell script that checks disk space and exits nonzero below a threshold.',2),
 ('code_generation','Wrap a callback-based JavaScript file API in a Promise with typed errors.',3),
 ('code_generation','Build a Go CLI that parses flags, validates input paths, and prints JSON output.',4),
 ('code_generation','Implement a resumable Python downloader that verifies the ETag before appending.',5),
 ('code_generation','Write a C function that returns the lower-bound insertion index in a sorted array.',3),
 ('code_generation','Convert XML records to JSON in Kotlin while handling missing optional elements.',4),
 ('code_generation','Create a thread-safe bounded cache in C# with expiration and eviction.',5),
 ('code_generation','Write a jq filter that extracts failed requests and groups them by service name.',2),
 ('code_generation','Implement a FastAPI endpoint that validates a date range and returns matching rows.',4),
 ('code_generation','Create a SQL upsert that increments inventory only when the SKU already exists.',3),
 ('code_generation','Write a C# CSV import routine that supports cancellation and records row errors.',4),
 ('code_generation','Implement a Python function that parses CSV text and returns the top five rows.',3),
 ('code_understanding','Explain what this recursive function returns for an empty list.',2),
 ('code_understanding','Find the race condition in this async cache implementation.',4),
 ('code_understanding','Summarize the control flow and error handling in this module.',3),
 ('code_understanding','Explain why this Java stream operation changes the original collection.',3),
 ('code_understanding','Trace ownership and borrowing through this Rust function.',4),
 ('code_understanding','What happens when this Python generator is closed before it is exhausted?',3),
 ('code_understanding','Explain how cancellation propagates through these nested Go contexts.',4),
 ('code_understanding','Identify the lock ordering that can deadlock these two worker threads.',4),
 ('code_understanding','Why can this React effect read an outdated value from its closure?',3),
 ('code_understanding','Walk through the result of this SQL transaction when the second update fails.',4),
 ('code_understanding','Explain which strings this regular expression accepts and rejects.',2),
 ('code_understanding','Review this retry loop and describe its stopping conditions.',3),
 ('code_understanding','Trace how this exception moves from the repository to the HTTP response.',4),
 ('code_understanding','Compare the behavior of these two implementations for an empty input.',2),
 ('code_understanding','Explain how this heap-based routine chooses the next item.',3),
 ('code_understanding','Find the error paths that are not handled by this REST endpoint.',4),
 ('code_understanding','Explain how this YAML loader chooses defaults when a key is absent.',2),
 ('code_understanding','What does this database query plan indicate about the missing index?',4),
 ('code_understanding','Summarize how this module synchronizes access to shared state.',3),
 ('code_understanding','Why does this test sometimes pass and sometimes fail based on the code shown?',4),
 ('code_understanding','Explain how this parser handles escaped quotes in a string literal.',3),
 ('code_understanding','What values can this TypeScript discriminated union represent?',2),
 ('code_understanding','Explain why this C pointer becomes invalid after the vector grows.',4),
 ('code_understanding','Summarize the state transitions in this payment processing function.',3),
 ('code_understanding','Which branch handles an empty response in this client code?',2),
 ('technical_design','Design a rate limiter for a multi-region API.',3),
 ('technical_design','Compare queue-based and event-stream architectures for this workload.',4),
 ('technical_design','Propose a migration plan from a monolith to bounded services.',5),
 ('technical_design','Design an API for uploading large files with resumable chunks.',4),
 ('technical_design','Plan a zero-downtime migration from integer IDs to UUIDs.',5),
 ('technical_design','Design an authentication service that supports key rotation and revocation.',5),
 ('technical_design','Propose a cache invalidation strategy for product data updated by several services.',4),
 ('technical_design','Design feature flags with gradual rollout and emergency rollback.',4),
 ('technical_design','Plan a durable workflow engine for jobs that can run for several hours.',5),
 ('technical_design','Design observability for correlating traces, logs, and metrics across services.',4),
 ('technical_design','Propose tenant isolation and quota enforcement for a shared database.',5),
 ('technical_design','Design idempotent webhook delivery with retries and per-customer ordering.',4),
 ('technical_design','Plan a secrets rotation system for hundreds of deployed services.',5),
 ('technical_design','Design a search index update pipeline that tolerates duplicate events.',4),
 ('technical_design','Compare database-per-tenant and shared-schema tenancy for this product.',4),
 ('technical_design','Design a plugin interface that keeps third-party extensions isolated.',5),
 ('technical_design','Plan a service deployment strategy with canary traffic and fast rollback.',4),
 ('technical_design','Design a notification system with user preferences and delivery receipts.',4),
 ('technical_design','Propose a schema for versioned API contracts used by several teams.',3),
 ('technical_design','Design a resilient payment flow that handles duplicate and delayed callbacks.',5),
 ('analytical_reasoning','Given these constraints, determine the minimum number of workers.',3),
 ('analytical_reasoning','Check this proof for a gap and derive the corrected bound.',5),
 ('analytical_reasoning','Reason through the possible outcomes of this probability puzzle.',4),
 ('analytical_reasoning','A job takes 8 minutes on one worker; estimate time for 4 independent workers.',2),
 ('analytical_reasoning','Compare the asymptotic costs of sorting first versus scanning each query.',3),
 ('analytical_reasoning','Determine whether these premises logically entail the stated conclusion.',4),
 ('analytical_reasoning','Calculate the expected value when a fair die is rolled until a six appears.',3),
 ('analytical_reasoning','Solve this two-equation system and show how the constraints affect the answer.',3),
 ('analytical_reasoning','How many distinct schedules satisfy these precedence constraints?',4),
 ('analytical_reasoning','Given this graph, identify the shortest path and justify why it is minimal.',3),
 ('analytical_reasoning','Estimate the service capacity needed for 1200 requests per second at 80 ms each.',3),
 ('analytical_reasoning','Infer the loop invariant needed to prove this algorithm terminates.',5),
 ('analytical_reasoning','How many test cases cover every pair of these six binary settings?',4),
 ('analytical_reasoning','Determine which variable most likely caused the observed change from these measurements.',4),
 ('analytical_reasoning','If each retry succeeds independently with probability 0.7, find success chance within three tries.',3),
 ('analytical_reasoning','Show whether this greedy choice always produces an optimal schedule.',5),
 ('analytical_reasoning','A cache hit rate rises from 70 percent to 84 percent; calculate the relative reduction in misses.',2),
 ('analytical_reasoning','Derive the maximum number of edges in a simple graph with these vertex constraints.',4),
 ('analytical_reasoning','Use the given truth table to determine which input combinations make the rule false.',3),
 ('analytical_reasoning','Find the smallest batch size that meets the throughput and memory limits.',4),
 ('writing','Rewrite this paragraph in a concise and friendly tone.',2),
 ('writing','Draft an announcement explaining a delayed product launch.',3),
 ('writing','Edit this report for clarity while preserving its claims.',3),
 ('writing','Write a polite reminder asking a teammate to review a pending change.',2),
 ('writing','Create release notes for a version that adds export and fixes login errors.',3),
 ('writing','Summarize this meeting into decisions, owners, and next steps.',3),
 ('writing','Draft an apology to a customer whose invoice was charged twice.',3),
 ('writing','Turn these technical notes into a short onboarding guide.',4),
 ('writing','Write a neutral status update about an investigation that is still in progress.',3),
 ('writing','Suggest five clear titles for an article about reducing cloud costs.',2),
 ('writing','Make this support response warmer without promising an unconfirmed fix.',3),
 ('writing','Draft a project proposal with goals, scope, and risks.',4),
 ('writing','Convert this list of product features into a concise announcement.',3),
 ('writing','Write a short thank-you note to a volunteer event organizer.',2),
 ('writing','Improve the transitions in this essay while keeping the author voice.',4),
 ('writing','Prepare a FAQ answer explaining how account deletion works.',3),
 ('writing','Create a one-paragraph executive summary from these findings.',3),
 ('writing','Rewrite this warning so a new user can understand the consequence.',2),
 ('writing','Draft a cover letter for a software role using these experience notes.',4),
 ('writing','Edit this policy paragraph to remove ambiguity without changing meaning.',4),
 ('factual_lookup','What does HTTP status code 429 mean?',1),
 ('factual_lookup','When was the first edition of The Hobbit published?',1),
 ('factual_lookup','Name the capital city of Portugal.',1),
 ('factual_lookup','What is the chemical symbol for potassium?',1),
 ('factual_lookup','How many sides does a hexagon have?',1),
 ('factual_lookup','Which planet is known for its prominent ring system?',1),
 ('factual_lookup','What does CPU stand for?',1),
 ('factual_lookup','At standard atmospheric pressure, what is the boiling point of water in Celsius?',1),
 ('factual_lookup','Who wrote the play Hamlet?',1),
 ('factual_lookup','What is the largest ocean on Earth?',1),
 ('factual_lookup','Which data format uses key-value pairs and arrays in text?',1),
 ('factual_lookup','How many bits are in one byte?',1),
 ('factual_lookup','What is the primary language spoken in Brazil?',1),
 ('factual_lookup','Which organ pumps blood through the human body?',1),
 ('factual_lookup','What is the square root of 144?',1),
 ('factual_lookup','Which protocol is commonly used to resolve domain names?',1),
 ('factual_lookup','What is the freezing point of water in Celsius at standard pressure?',1),
 ('factual_lookup','Which continent contains the Sahara Desert?',1),
 ('factual_lookup','What does the HTML abbreviation stand for?',1),
 ('factual_lookup','How many degrees are in a right angle?',1),
 ('general','Hi, can you help me with something?',1),
 ('general','Give me a few ideas for a weekend activity.',2),
 ('general','Thanks, that answers my question.',1),
 ('general','How is your day going so far?',1),
 ('general','I am not sure what to ask yet.',1),
 ('general','Can you explain that in simpler words?',2),
 ('general','Could you help me think of something fun to do this weekend?',2),
 ('general','What are some easy ways to stay organized?',2),
 ('general','I need a quick break from work. Any suggestions?',2),
 ('general','Could you help me brainstorm a name for a book club?',2),
 ('general','That sounds good, thank you.',1),
 ('general','Can we start with the basics?',1),
 ('general','Give me a simple dinner idea for tonight.',1),
 ('general','I have a few minutes before my meeting.',1),
 ('general','What should I keep in mind when learning a new skill?',2),
 ('general','Could you give me a short overview of your capabilities?',1),
 ('general','I would like to think through an idea out loud.',2),
 ('general','Can you make that answer a little shorter?',1),
 ('general','Suggest a low-cost activity for a rainy afternoon.',2),
 ('general','I am looking for a fresh perspective on my routine.',2),
];
data = pd.DataFrame(rows, columns=['request_type','query','complexity'])
data['group'] = [f'seed-{i}' for i in range(len(data))]
data['source'] = 'nasiko_seed_v2'
data['id'] = data['group']
extra_path = os.environ.get('LLMROUTER_LABELLED_DATA')
if extra_path:
    extra = pd.read_json(extra_path, lines=True)
    required = {'id','query','request_type','complexity','group','source'}
    assert required <= set(extra.columns), f'missing columns: {required-set(extra.columns)}'
    if 'context' not in extra: extra['context'] = ''
    data = pd.concat([data, extra], ignore_index=True, sort=False).fillna({'context':''})
else:
    data['context'] = ''
assert set(data['request_type']) == set(LABELS), 'Every one of the seven request types must be represented.'
assert data['complexity'].between(1,5).all(), 'Complexity labels must be in 1..=5.'
assert data[['id','query','group','source']].notna().all().all(), 'id, query, group, and source must be present.'
assert not data.duplicated(['source','id']).any(), 'Example ids must be unique within each source dataset.'
data['query'] = data['query'].astype(str).str.strip()
assert data['query'].ne('').all(), 'Queries must not be empty.'
data['split_group'] = data['source'].astype(str) + '::' + data['group'].astype(str)
assert data.groupby('split_group')['request_type'].nunique().max() == 1, 'A paraphrase group must have one request_type label.'
group_counts = data.groupby('request_type')['split_group'].nunique().reindex(LABELS, fill_value=0)
if (group_counts < 5).any():
    raise ValueError(f'Need at least five independent groups per request type for a stratified 5-way split: {group_counts.to_dict()}')
print('rows', len(data), 'sources', data.source.nunique(), 'groups', data.split_group.nunique())


rows 155 sources 1 groups 155


In [19]:
# Keep complete sources together when enough exist; otherwise use stratified paraphrase-group folds.
if data.source.nunique() >= 3:
    sources = data[['source']].drop_duplicates().reset_index(drop=True)
    split1 = GroupShuffleSplit(n_splits=1, test_size=.15, random_state=73)
    rem_idx, test_idx = next(split1.split(sources, groups=sources.source))
    test_sources = set(sources.iloc[test_idx].source)
    remaining = data[~data.source.isin(test_sources)]
    rem_sources = remaining[['source']].drop_duplicates().reset_index(drop=True)
    split2 = GroupShuffleSplit(n_splits=1, test_size=.1765, random_state=74)
    train_source_idx, val_source_idx = next(split2.split(rem_sources, groups=rem_sources.source))
    val_sources = set(rem_sources.iloc[val_source_idx].source)
    train = data[~data.source.isin(test_sources | val_sources)].copy()
    valid = data[data.source.isin(val_sources)].copy()
    test = data[data.source.isin(test_sources)].copy()
else:
    print('Fewer than three independent sources: using stratified paraphrase-group splits; source-level generalization is unmeasured.')
    split1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=73)
    rem_idx, test_idx = next(split1.split(data, data['request_type'], groups=data['split_group']))
    remaining, test = data.iloc[rem_idx].copy(), data.iloc[test_idx].copy()
    split2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=74)
    train_idx, val_idx = next(split2.split(remaining, remaining['request_type'], groups=remaining['split_group']))
    train, valid = remaining.iloc[train_idx].copy(), remaining.iloc[val_idx].copy()
train=train.reset_index(drop=True); valid=valid.reset_index(drop=True); test=test.reset_index(drop=True)
assert not (set(train.split_group)&set(valid.split_group) or set(train.split_group)&set(test.split_group) or set(valid.split_group)&set(test.split_group))
for split_name, split in [('train',train),('validation',valid),('test',test)]:
    missing = set(LABELS) - set(split['request_type'])
    if missing: raise ValueError(f'{split_name} split is missing request types {sorted(missing)}; add more independent labeled groups/sources.')
print('split sizes', len(train), len(valid), len(test), 'sources', train.source.nunique(), valid.source.nunique(), test.source.nunique())


Fewer than three independent sources: using stratified paraphrase-group splits; source-level generalization is unmeasured.
split sizes 93 31 31 sources 1 1 1


In [20]:
def embed_text(text):
    tokens=TOKENIZER(text,return_tensors='pt',truncation=True,max_length=4096)
    with torch.inference_mode():
        hidden=ENCODER(**tokens).last_hidden_state
        mask=tokens['attention_mask'].unsqueeze(-1).to(hidden.dtype)
        pooled=(hidden*mask).sum(dim=1)/mask.sum(dim=1).clamp(min=1)
    return pooled.cpu().numpy().astype(np.float32).reshape(-1)
def format_input_text(query, context=''):
    text = str(query)
    if context:
        text += '\nContext:\n' + str(context)
    return text
def embed_frame(frame):
    texts = [format_input_text(query, context) for query, context in frame[['query','context']].itertuples(index=False, name=None)]
    return np.vstack([embed_text(text) for text in texts])
t0=time.perf_counter(); X_train=embed_frame(train); X_val=embed_frame(valid); X_test=embed_frame(test); embedding_seconds=time.perf_counter()-t0
print(X_train.shape, 'embedding_seconds', round(embedding_seconds,3))


(93, 768) embedding_seconds 391.273


In [21]:
# Tune k and uniform/distance weighting using grouped CV on training data only.
# Macro F1 is primary; ties prefer lower k, then uniform weighting (less per-query work).
train_group_counts=train.groupby('request_type')['split_group'].nunique()
n_splits=min(5, int(train_group_counts.min()))
if n_splits < 2: raise ValueError('Need at least two independent training groups per request type for grouped cross-validation.')
cv=StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=75); candidates=[]
for k in (1,3,5,7,9):
  if k > len(train): continue
  for weights in ('uniform','distance'):
    scores=[]
    for a,b in cv.split(X_train, train['request_type'], groups=train['split_group']):
      model=KNeighborsClassifier(n_neighbors=min(k,len(a)),weights=weights,metric='cosine',algorithm='brute')
      model.fit(X_train[a],train.request_type.iloc[a]); scores.append(f1_score(train.request_type.iloc[b],model.predict(X_train[b]),labels=LABELS,average='macro',zero_division=0))
    candidates.append((float(np.mean(scores)), k, weights))
cv_results=pd.DataFrame(candidates,columns=['macro_f1','k','weights'])
cv_results['cost_rank']=cv_results.weights.map({'uniform':0,'distance':1})
cv_results=cv_results.sort_values(['macro_f1','k','cost_rank'],ascending=[False,True,True]).drop(columns='cost_rank')
display(cv_results); best_f1,best_k,best_weights=cv_results.iloc[0].tolist(); best_k=int(best_k)
type_model=KNeighborsClassifier(n_neighbors=min(best_k,len(train)),weights=best_weights,metric='cosine',algorithm='brute').fit(X_train,train.request_type)
complexity_model=KNeighborsRegressor(n_neighbors=min(best_k,len(train)),weights=best_weights,metric='cosine',algorithm='brute').fit(X_train,train.complexity.astype(float))


,macro_f1,k,weights
6,0.900522,7,uniform
7,0.890000,7,distance
3,0.876423,3,distance
9,0.874376,9,distance
0,0.873741,1,uniform
1,0.873741,1,distance
5,0.872902,5,distance
2,0.864995,3,uniform
4,0.854694,5,uniform
8,0.838753,9,uniform


In [22]:
# Calibrate max vote probability on validation data; report only once on held-out test.
def vote_confidence(model, X): return model.predict_proba(X).max(axis=1)
MIN_CONFIDENCE = 0.85  # Keep evaluation aligned with the Rust runtime default.
MIN_CALIBRATION_GROUPS = 30  # Small validation sets cannot support reliable calibration.
if len(valid) == 0 or len(test) == 0:
    raise ValueError('Validation and test splits must both contain examples.')
if len(test) < 30:
    print(f'WARNING: held-out test has only {len(test)} examples; metrics are exploratory, not a performance claim.')
val_raw=vote_confidence(type_model,X_val)
val_correct=(type_model.predict(X_val)==valid['request_type']).astype(float)
if (valid['split_group'].nunique() >= MIN_CALIBRATION_GROUPS and np.unique(val_raw).size >= 2 and np.unique(val_correct).size >= 2):
    calibrator=IsotonicRegression(y_min=0.,y_max=1.,out_of_bounds='clip').fit(val_raw,val_correct)
    calibration_method='isotonic_validation'
else:
    print(f'WARNING: only {valid.split_group.nunique()} independent validation groups; setting confidence to zero so runtime falls back to regex.')
    calibrator=IsotonicRegression(y_min=0.,y_max=1.,out_of_bounds='clip').fit([0.,1.],[0.,0.])
    calibration_method='zero_confidence_insufficient_validation'
pred=type_model.predict(X_test); pred_complexity=np.clip(np.rint(complexity_model.predict(X_test)),1,5).astype(int)
raw_test=vote_confidence(type_model,X_test); confidence=calibrator.predict(raw_test)
print(classification_report(test['request_type'],pred,labels=LABELS,zero_division=0))
print('macro_f1',f1_score(test['request_type'],pred,labels=LABELS,average='macro',zero_division=0))
print('complexity_mae',mean_absolute_error(test['complexity'],pred_complexity))
print('request_type_accuracy',accuracy_score(test['request_type'],pred),'mean_calibrated_confidence',float(np.mean(confidence)))
print('calibration_method',calibration_method)
print('fallback_rate_at_confidence_0.85',float(np.mean(confidence < MIN_CONFIDENCE)))
correct=(pred==test['request_type'].to_numpy()); ece=0.
for lo in np.linspace(0,0.9,10):
    mask=(confidence>=lo)&((confidence<lo+0.1) if lo<0.9 else (confidence<=1.))
    if mask.any(): ece+=mask.mean()*abs(float(correct[mask].mean())-float(confidence[mask].mean()))
print('expected_calibration_error_10_bins',ece)
latencies=[]
for query, context in test[['query','context']].itertuples(index=False, name=None):
    text=format_input_text(query,context)
    start=time.perf_counter(); vec=embed_text(text).reshape(1,-1); type_model.predict(vec); latencies.append((time.perf_counter()-start)*1000)
print('latency_ms_p50',float(np.percentile(latencies,50)) if latencies else None,'p95',float(np.percentile(latencies,95)) if latencies else None)


                      precision    recall  f1-score   support

     code_generation       1.00      1.00      1.00         8
  code_understanding       0.44      1.00      0.62         4
    technical_design       1.00      1.00      1.00         4
analytical_reasoning       1.00      0.20      0.33         5
             writing       0.75      1.00      0.86         3
      factual_lookup       1.00      0.50      0.67         4
             general       1.00      1.00      1.00         3

            accuracy                           0.81        31
           macro avg       0.88      0.81      0.78        31
        weighted avg       0.90      0.81      0.79        31

macro_f1 0.7817896389324961
complexity_mae 0.6129032258064516
request_type_accuracy 0.8064516129032258 mean_calibrated_confidence 0.9119544592030361
calibration_method isotonic_validation
fallback_rate_at_confidence_0.85 0.0967741935483871
expected_calibration_error_10_bins 0.04743833017077805
latency_ms_p50 2337.

In [23]:
# Export exact model metadata and dataset hashes for reproducibility. Never bundle source text.
def file_sha256(path):
  h=hashlib.sha256();
  with open(path,'rb') as f:
    for chunk in iter(lambda:f.read(1<<20),b''): h.update(chunk)
  return h.hexdigest()
dataset_hashes={'combined_labelled_rows_sha256':hashlib.sha256(data[['id','query','context','request_type','complexity','group','source']].to_json(orient='records',force_ascii=False).encode()).hexdigest()}
if extra_path: dataset_hashes[Path(extra_path).name]=file_sha256(extra_path)
metadata={'embedding_model_id':EMBEDDING_MODEL_ID,'embedding_revision':EMBEDDING_REVISION,'package':'llmrouter-lib==0.3.1','preprocessing':'query + newline Context + context; pinned Longformer last_hidden_state attention-mask mean pooling (4096 token cap)','metric':'cosine','k':best_k,'weights':best_weights,'cv_macro_f1':float(best_f1),'calibration_method':calibration_method,'min_confidence':MIN_CONFIDENCE,'random_seeds':{'test_split':73,'validation_split':74,'grouped_cv':75},'dataset_hashes':dataset_hashes,'split_counts':{'train':len(train),'validation':len(valid),'test':len(test)}}
bundle={'schema_version':1,'request_type_knn':type_model,'request_type_labels':train.request_type.to_numpy(),'complexity_knn':complexity_model,'complexity_labels':train.complexity.astype(float).to_numpy(),'confidence_calibrator':calibrator,'metadata':metadata}
OUT.parent.mkdir(parents=True,exist_ok=True); joblib.dump(bundle,OUT)
print('exported',OUT,metadata)


exported llm-router/artifacts/request_classifier.joblib {'embedding_model_id': 'allenai/longformer-base-4096', 'embedding_revision': '301e6a42cb0d9976a6d6a26a079fef81c18aa895', 'package': 'llmrouter-lib==0.3.1', 'preprocessing': 'query + newline Context + context; pinned Longformer last_hidden_state attention-mask mean pooling (4096 token cap)', 'metric': 'cosine', 'k': 7, 'weights': 'uniform', 'cv_macro_f1': 0.9005215419501134, 'calibration_method': 'isotonic_validation', 'min_confidence': 0.85, 'random_seeds': {'test_split': 73, 'validation_split': 74, 'grouped_cv': 75}, 'dataset_hashes': {'combined_labelled_rows_sha256': 'f1fd33f32680f17441f831e5bc04301a12161bb645504065af5067e386036987'}, 'split_counts': {'train': 93, 'validation': 31, 'test': 31}}
